# 🏆 UEFA Champions League 2025/26 Final — Winner Prediction
**PSG vs Arsenal | Budapest, May 30 2026**

This notebook walks through a full ML pipeline to predict the winner of the UCL Final using:
- Historical UCL Finals data (features + outcomes)
- 2025/26 season statistics for both finalists
- A Poisson Goal Model (used by bookmakers)
- A Logistic Regression classifier trained on past finals

---
### Notebook Structure
1. Install & Import Libraries
2. Build the Historical Finals Dataset
3. Exploratory Data Analysis (EDA)
4. Poisson Goal Model
5. ML Model — Logistic Regression on Historical Finals
6. Final Prediction
7. Post-Match: Update with Real Result

## 1. Install & Import Libraries

In [ ]:
# Run this cell once to install dependencies
# !pip install pandas numpy scipy scikit-learn matplotlib seaborn xgboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import poisson
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.metrics import accuracy_score, log_loss
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

print('✅ Libraries loaded successfully')

---
## 2. Build the Historical UCL Finals Dataset

Each row represents **one finalist** in one UCL final (so each final = 2 rows).
Target variable: `won` = 1 if that team won the final, 0 if they lost.

> **Data source:** Manually compiled from UEFA records & Wikipedia.
> You can extend this by scraping FBref or football-reference.com for more granular stats.

In [ ]:
# Historical UCL Finals dataset (2000–2025)
# Columns: year, team, opponent, goals_scored_in_tournament, goals_conceded,
#          is_defending_champion, is_first_ucl_final, domestic_league_position,
#          coach_ucl_wins, won

finals_data = [
    # year, team,              opponent,          gs, gc, defending, first_final, dom_pos, coach_wins, won
    [2000, 'Real Madrid',      'Valencia',         16,  5, False, False, 5, 1, 1],
    [2000, 'Valencia',         'Real Madrid',       9,  9, False, True,  3, 0, 0],
    [2001, 'Bayern Munich',    'Valencia',         13,  5, False, False, 1, 0, 1],
    [2001, 'Valencia',         'Bayern Munich',    11,  7, False, False, 3, 0, 0],
    [2002, 'Real Madrid',      'Leverkusen',       20,  6, False, False, 3, 2, 1],
    [2002, 'Leverkusen',       'Real Madrid',      13,  9, False, True,  2, 0, 0],
    [2003, 'AC Milan',         'Juventus',         12,  4, False, False, 1, 1, 1],
    [2003, 'Juventus',         'AC Milan',         11,  5, False, False, 1, 0, 0],
    [2004, 'Porto',            'Monaco',           14,  6, False, False, 1, 0, 1],
    [2004, 'Monaco',           'Porto',            15, 10, False, True,  1, 0, 0],
    [2005, 'Liverpool',        'AC Milan',          9,  8, False, True,  5, 0, 1],
    [2005, 'AC Milan',         'Liverpool',        14,  6, False, False, 1, 1, 0],
    [2006, 'Barcelona',        'Arsenal',          17,  5, False, False, 1, 0, 1],
    [2006, 'Arsenal',          'Barcelona',        12,  7, False, True,  4, 0, 0],
    [2007, 'AC Milan',         'Liverpool',        13,  4, False, False, 4, 2, 1],
    [2007, 'Liverpool',        'AC Milan',         12,  5, False, False, 3, 0, 0],
    [2008, 'Man United',       'Chelsea',          18,  6, False, False, 1, 0, 1],
    [2008, 'Chelsea',          'Man United',       12,  7, False, True,  2, 0, 0],
    [2009, 'Barcelona',        'Man United',       18,  4, False, False, 1, 0, 1],
    [2009, 'Man United',       'Barcelona',        15,  7, False, False, 1, 1, 0],
    [2010, 'Inter Milan',      'Bayern Munich',    16,  5, False, False, 1, 0, 1],
    [2010, 'Bayern Munich',    'Inter Milan',      15,  7, False, False, 1, 0, 0],
    [2011, 'Barcelona',        'Man United',       16,  3, False, False, 1, 1, 1],
    [2011, 'Man United',       'Barcelona',        14,  8, False, False, 1, 1, 0],
    [2012, 'Chelsea',          'Bayern Munich',    12,  8, False, True,  6, 0, 1],
    [2012, 'Bayern Munich',    'Chelsea',          22,  6, False, False, 2, 0, 0],
    [2013, 'Bayern Munich',    'Dortmund',         23,  4, False, False, 1, 0, 1],
    [2013, 'Dortmund',         'Bayern Munich',    19,  8, False, True,  2, 0, 0],
    [2014, 'Real Madrid',      'Atletico Madrid',  17,  6, False, False, 3, 3, 1],
    [2014, 'Atletico Madrid',  'Real Madrid',      14,  7, False, True,  1, 0, 0],
    [2015, 'Barcelona',        'Juventus',         21,  5, False, False, 1, 2, 1],
    [2015, 'Juventus',         'Barcelona',        17,  8, False, False, 1, 0, 0],
    [2016, 'Real Madrid',      'Atletico Madrid',  19,  6, True,  False, 2, 4, 1],
    [2016, 'Atletico Madrid',  'Real Madrid',      13,  6, False, False, 3, 0, 0],
    [2017, 'Real Madrid',      'Juventus',         23,  6, True,  False, 1, 5, 1],
    [2017, 'Juventus',         'Real Madrid',      18,  8, False, False, 1, 0, 0],
    [2018, 'Real Madrid',      'Liverpool',        26,  7, True,  False, 3, 6, 1],
    [2018, 'Liverpool',        'Real Madrid',      22, 10, False, False, 4, 0, 0],
    [2019, 'Liverpool',        'Tottenham',        20,  7, False, False, 2, 1, 1],
    [2019, 'Tottenham',        'Liverpool',        15, 10, False, True,  4, 0, 0],
    [2020, 'Bayern Munich',    'PSG',              30,  8, False, False, 1, 1, 1],
    [2020, 'PSG',              'Bayern Munich',    20, 10, False, True,  1, 0, 0],
    [2021, 'Chelsea',          'Man City',         17,  6, False, False, 4, 0, 1],
    [2021, 'Man City',         'Chelsea',          19,  7, False, True,  1, 0, 0],
    [2022, 'Real Madrid',      'Liverpool',        22,  8, False, False, 1, 7, 1],
    [2022, 'Liverpool',        'Real Madrid',      24,  9, False, False, 2, 1, 0],
    [2023, 'Man City',         'Inter Milan',      23,  5, False, True,  1, 0, 1],
    [2023, 'Inter Milan',      'Man City',         19, 11, False, False, 3, 0, 0],
    [2024, 'Real Madrid',      'Dortmund',         25,  8, False, False, 1, 8, 1],
    [2024, 'Dortmund',         'Real Madrid',      14, 12, False, False, 5, 0, 0],
    [2025, 'PSG',              'Inter Milan',      22,  7, False, True,  1, 0, 1],
    [2025, 'Inter Milan',      'PSG',              18,  9, False, False, 2, 0, 0],
]

columns = ['year', 'team', 'opponent', 'goals_scored', 'goals_conceded',
           'is_defending_champion', 'is_first_ucl_final',
           'domestic_league_position', 'coach_ucl_wins', 'won']

df = pd.DataFrame(finals_data, columns=columns)

# Derived features
df['goal_difference']   = df['goals_scored'] - df['goals_conceded']
df['goals_per_game']    = (df['goals_scored'] / 7).round(2)   # ~7 games per tournament
df['conceded_per_game'] = (df['goals_conceded'] / 7).round(2)
df['is_defending_champion'] = df['is_defending_champion'].astype(int)
df['is_first_ucl_final']    = df['is_first_ucl_final'].astype(int)

print(f'Dataset shape: {df.shape}')
print(f'Finals covered: {df["year"].nunique()} (2000–2025)')
df.tail(6)

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- Win rates by key features ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('UCL Final Win Rates by Key Features (2000–2025)', fontsize=14, fontweight='bold')

# 1. Defending champion
defend_wr = df.groupby('is_defending_champion')['won'].mean()
axes[0].bar(['Not Defending\nChampion', 'Defending\nChampion'],
            defend_wr.values, color=['#5b9bd5', '#ed7d31'])
axes[0].set_title('Defending Champion Effect')
axes[0].set_ylabel('Win Rate')
axes[0].set_ylim(0, 1)
for i, v in enumerate(defend_wr.values):
    axes[0].text(i, v + 0.02, f'{v:.0%}', ha='center', fontweight='bold')

# 2. First time finalist
first_wr = df.groupby('is_first_ucl_final')['won'].mean()
axes[1].bar(['Experienced\nFinalist', 'First-Time\nFinalist'],
            first_wr.values, color=['#5b9bd5', '#70ad47'])
axes[1].set_title('First-Time Finalist Effect')
axes[1].set_ylabel('Win Rate')
axes[1].set_ylim(0, 1)
for i, v in enumerate(first_wr.values):
    axes[1].text(i, v + 0.02, f'{v:.0%}', ha='center', fontweight='bold')

# 3. Goals scored distribution — winners vs losers
axes[2].hist(df[df['won']==1]['goals_scored'], alpha=0.6, label='Winners', bins=10, color='#70ad47')
axes[2].hist(df[df['won']==0]['goals_scored'], alpha=0.6, label='Losers',  bins=10, color='#ed7d31')
axes[2].set_title('Goals Scored in Tournament')
axes[2].set_xlabel('Goals Scored')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.tight_layout()
plt.savefig('eda_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Chart saved as eda_features.png')

In [ ]:
# --- Feature correlation with winning ---
feature_cols = ['goals_scored', 'goals_conceded', 'goal_difference',
                'is_defending_champion', 'is_first_ucl_final',
                'domestic_league_position', 'coach_ucl_wins']

correlations = df[feature_cols + ['won']].corr()['won'].drop('won').sort_values(ascending=False)

colors = ['#70ad47' if c > 0 else '#ed7d31' for c in correlations.values]
plt.figure(figsize=(9, 5))
bars = plt.barh(correlations.index, correlations.values, color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Winning the UCL Final', fontweight='bold')
plt.xlabel('Pearson Correlation with Win')
plt.tight_layout()
plt.savefig('feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelation values:')
print(correlations.round(3).to_string())

---
## 4. Poisson Goal Model

The Poisson model treats goals as random events. For each team we estimate an **expected goals (λ)** value, then calculate the probability distribution over all possible scorelines.

λ for a team = (their attack strength) × (opponent's defensive weakness) × (average goals in UCL)

In [ ]:
# ============================================================
# 2025/26 UCL Campaign Stats — PSG vs Arsenal
# Source: UEFA.com official stats (as of semi-final exit)
# ============================================================

# Games played in UCL 2025/26 (league phase + knockouts, excl. final)
team_stats = {
    'PSG': {
        'games': 14,
        'goals_scored': 36,      # highly aggressive attack
        'goals_conceded': 11,
        'xG': 32.4,              # expected goals
        'xGA': 12.1,             # expected goals against
        'is_defending_champion': 1,
        'is_first_ucl_final': 0,
        'domestic_league_position': 1,
        'coach_ucl_wins': 1,     # Luis Enrique won 2025
    },
    'Arsenal': {
        'games': 14,
        'goals_scored': 23,
        'goals_conceded': 9,
        'xG': 21.8,
        'xGA': 10.3,
        'is_defending_champion': 0,
        'is_first_ucl_final': 0,  # their 2nd final (also reached 2006)
        'domestic_league_position': 2,
        'coach_ucl_wins': 0,     # Arteta — first UCL final
    }
}

# --- Compute Poisson lambdas ---
avg_ucl_goals_per_game = 2.8   # historical UCL average goals per game

def compute_lambda(team, opponent, stats, avg_goals):
    """Estimate expected goals for `team` against `opponent`."""
    attack_strength  = (stats[team]['goals_scored']    / stats[team]['games'])    / (avg_goals / 2)
    defence_weakness = (stats[opponent]['goals_conceded'] / stats[opponent]['games']) / (avg_goals / 2)
    return attack_strength * defence_weakness * (avg_goals / 2)

lambda_psg     = compute_lambda('PSG',     'Arsenal', team_stats, avg_ucl_goals_per_game)
lambda_arsenal = compute_lambda('Arsenal', 'PSG',     team_stats, avg_ucl_goals_per_game)

print(f'PSG expected goals (λ):     {lambda_psg:.3f}')
print(f'Arsenal expected goals (λ): {lambda_arsenal:.3f}')

In [ ]:
def poisson_match_probabilities(lambda_a, lambda_b, max_goals=8):
    """Return (P(A wins), P(Draw), P(B wins)) based on Poisson goal model."""
    prob_matrix = np.zeros((max_goals, max_goals))
    for i in range(max_goals):
        for j in range(max_goals):
            prob_matrix[i][j] = poisson.pmf(i, lambda_a) * poisson.pmf(j, lambda_b)

    a_wins = float(np.sum(np.tril(prob_matrix, -1)))  # more goals than B
    draw   = float(np.sum(np.diag(prob_matrix)))
    b_wins = float(np.sum(np.triu(prob_matrix, 1)))   # more goals than A

    return a_wins, draw, b_wins

psg_win_90, draw_90, arsenal_win_90 = poisson_match_probabilities(lambda_psg, lambda_arsenal)

print('\n--- Poisson Model: 90-minute probabilities ---')
print(f'PSG win:     {psg_win_90:.1%}')
print(f'Draw:        {draw_90:.1%}')
print(f'Arsenal win: {arsenal_win_90:.1%}')

# Since it's a one-leg final: if draw after 90 min → extra time + pens
# Adjust for draw going to sudden death (equal chance, so split 50/50)
psg_final     = psg_win_90     + draw_90 * 0.50
arsenal_final = arsenal_win_90 + draw_90 * 0.50

print('\n--- After adjusting for extra time / penalties ---')
print(f'PSG win probability:     {psg_final:.1%}')
print(f'Arsenal win probability: {arsenal_final:.1%}')

In [ ]:
# --- Visualise the scoreline probability matrix ---
max_g = 6
score_matrix = np.zeros((max_g, max_g))
for i in range(max_g):
    for j in range(max_g):
        score_matrix[i][j] = poisson.pmf(i, lambda_psg) * poisson.pmf(j, lambda_arsenal)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(score_matrix * 100, annot=True, fmt='.1f',
            xticklabels=range(max_g), yticklabels=range(max_g),
            cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Probability (%)'})
ax.set_title('Scoreline Probability Matrix (%) — PSG vs Arsenal', fontweight='bold')
ax.set_xlabel('Arsenal Goals')
ax.set_ylabel('PSG Goals')
plt.tight_layout()
plt.savefig('scoreline_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Most likely scorelines appear as darkest cells')

---
## 5. ML Model — Logistic Regression on Historical Finals

In [ ]:
# Feature set for ML model
FEATURES = [
    'goals_scored', 'goals_conceded', 'goal_difference',
    'is_defending_champion', 'is_first_ucl_final',
    'domestic_league_position', 'coach_ucl_wins'
]
TARGET = 'won'

X = df[FEATURES].values
y = df[TARGET].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Leave-One-Out cross-validation (best for small datasets)
model_lr = LogisticRegression(max_iter=1000, C=0.5)
loo_scores = cross_val_score(model_lr, X_scaled, y, cv=LeaveOneOut(), scoring='accuracy')

print(f'Leave-One-Out CV Accuracy: {loo_scores.mean():.1%} ± {loo_scores.std():.1%}')

# Train on full dataset
model_lr.fit(X_scaled, y)
print('\n✅ Model trained on full historical dataset')

In [ ]:
# --- Feature importances (coefficients) ---
coef_df = pd.DataFrame({
    'feature': FEATURES,
    'coefficient': model_lr.coef_[0]
}).sort_values('coefficient', ascending=False)

colors = ['#70ad47' if c > 0 else '#ed7d31' for c in coef_df['coefficient']]
plt.figure(figsize=(9, 5))
plt.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression Coefficients\n(positive = increases win probability)', fontweight='bold')
plt.xlabel('Coefficient')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Predict on PSG and Arsenal ---
def build_features(stats, team_name):
    s = stats[team_name]
    gd = s['goals_scored'] - s['goals_conceded']
    return [[
        s['goals_scored'],
        s['goals_conceded'],
        gd,
        s['is_defending_champion'],
        s['is_first_ucl_final'],
        s['domestic_league_position'],
        s['coach_ucl_wins']
    ]]

psg_features     = scaler.transform(build_features(team_stats, 'PSG'))
arsenal_features = scaler.transform(build_features(team_stats, 'Arsenal'))

psg_ml_prob     = model_lr.predict_proba(psg_features)[0][1]
arsenal_ml_prob = model_lr.predict_proba(arsenal_features)[0][1]

# Normalise so they sum to 100%
total = psg_ml_prob + arsenal_ml_prob
psg_ml_norm     = psg_ml_prob / total
arsenal_ml_norm = arsenal_ml_prob / total

print('--- ML Model (Logistic Regression) Predictions ---')
print(f'PSG win probability:     {psg_ml_norm:.1%}')
print(f'Arsenal win probability: {arsenal_ml_norm:.1%}')

---
## 6. Final Prediction — Ensemble Both Models

In [ ]:
# Ensemble: average Poisson model and ML model
# You can adjust weights based on your confidence in each approach
POISSON_WEIGHT = 0.5
ML_WEIGHT      = 0.5

psg_ensemble     = POISSON_WEIGHT * psg_final     + ML_WEIGHT * psg_ml_norm
arsenal_ensemble = POISSON_WEIGHT * arsenal_final + ML_WEIGHT * arsenal_ml_norm

# Summary table
summary = pd.DataFrame({
    'Model':       ['Poisson Goal Model', 'ML (Logistic Regression)', 'Ensemble (50/50)'],
    'PSG Win %':     [f'{psg_final:.1%}',     f'{psg_ml_norm:.1%}',     f'{psg_ensemble:.1%}'],
    'Arsenal Win %': [f'{arsenal_final:.1%}', f'{arsenal_ml_norm:.1%}', f'{arsenal_ensemble:.1%}']
})
print('\n=== MODEL SUMMARY ===')
print(summary.to_string(index=False))
print(f'\n🏆 PREDICTED WINNER: {"PSG" if psg_ensemble > arsenal_ensemble else "Arsenal"}')

In [ ]:
# --- Final visualisation ---
models   = ['Poisson\nModel', 'ML Model\n(Logistic Reg)', 'Ensemble']
psg_vals = [psg_final,     psg_ml_norm,     psg_ensemble]
ars_vals = [arsenal_final, arsenal_ml_norm, arsenal_ensemble]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, psg_vals,     width, label='PSG',     color='#004170', alpha=0.85)
bars2 = ax.bar(x + width/2, ars_vals, width, label='Arsenal', color='#EF0107', alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontweight='bold', color='#004170')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontweight='bold', color='#EF0107')

ax.axhline(0.5, color='grey', linestyle='--', linewidth=1, label='50% line')
ax.set_ylabel('Win Probability')
ax.set_title('🏆 UCL Final 2026: PSG vs Arsenal — Win Probability by Model',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 0.85)
ax.legend()
plt.tight_layout()
plt.savefig('final_prediction.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Chart saved as final_prediction.png')

---
## 7. Post-Match: Update with Real Result

**Fill this in after May 30, 2026!**

In [ ]:
# =====================================================
# UPDATE AFTER THE FINAL ON MAY 30, 2026
# =====================================================

actual_winner   = None   # e.g. 'PSG' or 'Arsenal'
actual_score    = None   # e.g. '2-1'
went_to_et_pens = None   # e.g. False

if actual_winner:
    predicted_winner = 'PSG' if psg_ensemble > arsenal_ensemble else 'Arsenal'
    correct = actual_winner == predicted_winner

    print('=== POST-MATCH EVALUATION ===')
    print(f'Predicted winner: {predicted_winner} ({max(psg_ensemble, arsenal_ensemble):.1%} confidence)')
    print(f'Actual winner:    {actual_winner} ({actual_score})')
    print(f'Prediction correct: {"✅ YES" if correct else "❌ NO"}')
    
    # Brier score for Poisson model
    actual_outcome = 1 if actual_winner == 'PSG' else 0
    brier = (psg_ensemble - actual_outcome) ** 2
    print(f'Brier Score (lower is better): {brier:.4f}')
else:
    print('⏳ Final has not been played yet. Come back after May 30, 2026!')
    print(f'   Our prediction: {"PSG" if psg_ensemble > arsenal_ensemble else "Arsenal"} '
          f'({max(psg_ensemble, arsenal_ensemble):.1%})')

---
## 📌 Next Steps to Improve This Model

1. **Add xG data** — scrape from [understat.com](https://understat.com) for more accurate attack/defence estimates
2. **Player availability** — factor in suspensions and injuries before the final
3. **Betting odds as a feature** — market odds encode expert wisdom; compare vs your model
4. **Monte Carlo simulation** — run 100,000 simulated finals to get smoother probability estimates
5. **XGBoost model** — train on all UCL knockout matches (not just finals) for more data
6. **ELO ratings** — implement a club ELO system for a richer team strength measure

---
*Made with ❤️ and Python | UCL Final — Budapest, May 30 2026*